<div>

# **AI agent use Tavily**

<div dir=RTL>

# **התקנות / ספריות**

In [1]:
# התקנה
!pip install -q langgraph langchain-google-genai langchain-community tavily-python
# ספריות
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults

<div dir=RTL>

# **הגדרת מודל וכלים**

In [2]:
# מפתחות
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

# כלים
tools = [TavilySearchResults(max_results=3)]

# מודל
model = ChatGoogleGenerativeAI(
    model="gemini-pro-latest",
    google_api_key=os.environ["GOOGLE_API_KEY"]
).bind_tools(tools)

/tmp/ipykernel_2895/2400793546.py:6: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tools = [TavilySearchResults(max_results=3)]


<div dir=RTL>

# **הגדרת גרף**

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode

# צומת המודל
def call_model(state: MessagesState):
    return {"messages": [model.invoke(state["messages"])]}

# בדיקה: האם המודל ביקש להפעיל כלי?
def should_continue(state: MessagesState):
    return "tools" if state["messages"][-1].tool_calls else END

# בניית הגרף
graph = StateGraph(MessagesState)
graph.add_node("model", call_model)
graph.add_node("tools", ToolNode(tools))

graph.add_edge(START, "model")
graph.add_conditional_edges("model", should_continue)
graph.add_edge("tools", "model")

agent = graph.compile()

# הצגת הגרף
from IPython.display import Image, display
display(Image(agent.get_graph().draw_mermaid_png()))

<div dir=RTL>

# **הפעלת סוכן**

In [4]:
from langchain_core.messages import HumanMessage

# שאל את הסוכן שאלה
question = "מה החדשות הכי מעניינות בתחום ה-AI היום?"

collected_messages = []
for s in agent.stream({"messages": [HumanMessage(content=question)]}):
    # LangGraph's stream often yields dictionary where keys are node names
    # and values are the output of that node. We need to check these values.
    # The 'messages' are usually nested.

    # Check if 'model' node output messages
    if "model" in s and isinstance(s["model"], dict) and "messages" in s["model"]:
        collected_messages.extend(s["model"]["messages"])
    # Check if 'tools' node output messages (if applicable for debugging/tracing tool calls)
    elif "tools" in s and isinstance(s["tools"], dict) and "messages" in s["tools"]:
        collected_messages.extend(s["tools"]["messages"])
    # If the state itself contains messages (e.g., final state update might just have 'messages')
    elif "messages" in s:
        collected_messages.extend(s["messages"])

result = {"messages": collected_messages}

<div dir=RTL>

# **הדפסת התשובה**

In [5]:
# הדפסת תשובה
print(result["messages"][-1].content)

[{'type': 'text', 'text': 'הנה כמה מהחדשות המעניינות והחמות ביותר בתחום הבינה המלאכותית (AI) מהימים האחרונים, שמציגות קפיצת מדרגה משמעותית בתחרות בין ענקיות הטכנולוגיה:\n\n1. **השקת GPT-5.5 של OpenAI:** חברת OpenAI שחררה את המודל החדש והמתקדם שלה, **GPT-5.5**, צעד שנתפס כהתקדמות משמעותית בדרך ליצירת "אפליקציית-על" (Super App) מבוססת בינה מלאכותית. במקביל, מדווח כי החברה השלימה סבב גיוס חסר תקדים של 122 מיליארד דולר, שמעמיד את שווי החברה על למעלה מ-850 מיליארד דולר.\n\n2. **דליפה דרמטית באנתרופיק (Anthropic):** קבוצה לא מורשית הצליחה להשיג גישה ל-"Mythos" – כלי סייבר אקסקלוסיבי ומתקדם של חברת Anthropic (מפתחת מודל Claude), שהחברה הגדירה בעבר כמסוכן מדי לשחרור פומבי. סם אלטמן, מנכ"ל OpenAI, עקץ בתגובה וכינה את ההתנהלות של אנתרופיק סביב המודל כ"שיווק מבוסס הפחדה".\n\n3. **עסקת הענק של SpaceX ו-Cursor:** חברת החלל של אילון מאסק, SpaceX, משתפת פעולה עם סטארט-אפ הקידוד מבוסס ה-AI הפופולרי "Cursor", ולפי הדיווחים יש לה אופציה לרכוש את הסטארט-אפ בסכום עתק של 60 מיליארד דולר.\n\n4. **ההסתערות ש